# Apache Spark: distributed computing for data too big for one machine

_Apache Spark_

**A driver hands work to executors across a cluster, which crunch data partitions in parallel — lazily, with fault tolerance.**

One machine has a ceiling. A single computer has a fixed amount of memory and a fixed number of
       processor cores. When the data outgrows that ceiling, you cannot just keep asking for a bigger box &mdash;
       eventually the biggest box is still too small. Apache Spark's idea is to use many machines together,
       splitting one job across a whole cluster so they all work on a slice of the data at the same time.

       Three roles make this work:

---

This notebook is a practice scaffold for the **Apache Spark: distributed computing for data too big for one machine** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pyspark
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PySpark

### Step 1 — Start a Spark session in local mode

Every Spark program begins with a `SparkSession` — the handle that connects your code to the engine. In **local mode**, `master("local[*]")` tells Spark to treat every core on this one machine as an executor, which is perfect for Colab. The beautiful part: to point this same code at a real cluster later, you change only that one string.

In [ ]:
# Colab: !pip install pyspark   (run this in the notebook's setup cell first)
from pyspark.sql import SparkSession

# master("local[*]") = use every core on this machine as an "executor".
# Same code points at a real cluster later by changing only this string.
spark = (
    SparkSession.builder
        .master("local[*]")          # run locally, all cores (great for Colab)
        .appName("spark-intro")
        .getOrCreate()
)

print("Spark version:", spark.version)        # e.g. 3.5.x

### Step 2 — Build a small distributed DataFrame

A Spark `DataFrame` is a table split across the cluster. We create a tiny inline one here, but Spark treats these 5 rows exactly the way it would treat billions — the API is identical regardless of scale.

In [ ]:
# A tiny inline table. Spark treats this the same way it would treat billions of rows.
data = [("alice", "us", 3),
        ("bob",   "us", 1),
        ("carol", "uk", 4),
        ("dan",   "uk", 2),
        ("erin",  "de", 5)]

df = spark.createDataFrame(data, ["user", "country", "score"])

### Step 3 — Transformations are lazy; actions run the engine

This is Spark's defining trick. **Transformations** (like `groupBy`) only *describe* work — they build a plan and run nothing. An **action** (like `show` or `count`) is what actually triggers the engine to execute that plan. Watch: defining `by_country` is instant, but the `show()` after it is where the computation really happens.

In [ ]:
# ACTIONS: nothing above ran the engine; these trigger it.
df.show()                                      # prints the table (an action)
print("row count:", df.count())                # -> 5  (an action)

# A lazy TRANSFORMATION (builds plan, runs nothing yet)...
by_country = df.groupBy("country").sum("score")
by_country.show()                              # ...the show() action runs the plan

# How many PARTITIONS was the data split into? (parallel work units)
print("partitions:", df.rdd.getNumPartitions())   # e.g. 5 in local mode

### Step 4 — Always stop the session

A `SparkSession` holds onto cluster resources (memory, executor threads) until you release it. Calling `spark.stop()` frees them — good hygiene in any environment, and especially in a shared one like Colab.

In [ ]:
# ALWAYS STOP THE SESSION to release resources.
spark.stop()

## Visualize the data & results

_Where does each tool win as the data grows? Illustrative runtimes for pandas vs Spark across small / medium / huge data, showing Spark's fixed overhead loses on small data and only pays off at large scale._

### Step 1 — Write down a simple cost model

To see *where* each tool wins, we build a transparent (illustrative, not benchmarked) cost model. pandas runs purely in memory on one machine with no coordination overhead — but it breaks once the data exceeds that machine's RAM. Spark pays a fixed startup/shuffle cost, then spreads the per-megabyte work across many cores.

In [ ]:
import numpy as np

# ---- A simple, transparent cost MODEL (illustrative, not a benchmark) ----
# pandas: pure in-memory, no coordination overhead, single machine.
#   time = per-MB cost * size, but it BREAKS past one machine's RAM.
# Spark: a fixed startup/shuffle overhead, then per-MB work spread over many cores.
RAM_GB        = 16        # one machine's usable RAM
PANDAS_S_PER_MB = 0.001   # pandas in-memory cost per megabyte
SPARK_FIXED_S = 20.0      # start cluster + ship code + first shuffle
SPARK_S_PER_MB = 0.00045  # raw per-MB work...
SPARK_CORES   = 64        # ...spread across this many parallel cores

def pandas_time(gb):
    mb = gb * 1024
    if gb > RAM_GB:                       # too big for one machine
        return 1000.0                     # OOM / disk-thrash wall (cap for the chart)
    return round(PANDAS_S_PER_MB * mb, 2)

def spark_time(gb):
    mb = gb * 1024
    return round(SPARK_FIXED_S + SPARK_S_PER_MB * mb / SPARK_CORES, 2)

### Step 2 — Compare the two tools across three data sizes

Now we run the model at three scales — small (0.1 GB), medium (5 GB), and huge (200 GB) — and declare a winner at each. The story: pandas wins on small and medium data, but at 200 GB it blows past one machine's RAM (capped here at the 1000 s wall) and Spark finally pays off.

In [ ]:
sizes = [0.1, 5, 200]                     # GB: small, medium, huge
rows = []
for gb in sizes:
    p = pandas_time(gb)
    s = spark_time(gb)
    winner = "pandas" if p < s else "Spark"
    rows.append((gb, p, s, winner))
    print(f"{gb:6} GB   pandas {p:8}s   spark {s:8}s   -> {winner}")
# 0.1 GB   pandas     0.5 s   spark    20.5 s   -> pandas
#   5 GB   pandas    25.0 s   spark    33.0 s   -> pandas
# 200 GB   pandas  1000.0 s   spark    90.0 s   -> Spark   (pandas OOMs)

### Step 3 — Shape the numbers for plotting

Finally we flatten the results into the bar-chart inputs: one label and one runtime per (tool, size) pair, with pandas and Spark interleaved per size. These `labels`/`values` lists are exactly what a bar chart would consume.

In [ ]:
# Bars in the order plotted: pandas/Spark interleaved per size.
labels = [f"{w}\n{gb} GB" for gb in sizes for w in ("pandas", "Spark")]
values = [v for (_, p, s, _) in rows for v in (p, s)]
print(labels)
print(values)   # [0.5, 20.5, 25.0, 33.0, 1000.0, 90.0]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A colleague has a 400 MB CSV and proudly spins up a Spark cluster to groupBy-sum it, but it runs slower than your pandas one-liner. Why, and what should they do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the data size: 400 MB fits comfortably in the RAM of a single machine. — _Spark's whole advantage is spreading data that does not fit on one box; here it fits easily, so there is nothing to spread._
- Account for Spark's fixed overhead: starting the cluster, shipping code to executors, and the network shuffle a group-by triggers. — _On small data those fixed costs dominate the actual computation, so Spark loses to a tool with zero coordination cost._
- Recommend a single-machine tool: pandas, Polars, or DuckDB. — _They do the group-by-sum entirely in local memory with no cluster overhead, finishing far faster._

**Answer:** The data fits in RAM, so Spark's distribution machinery is pure overhead: it must start a cluster, ship the code to executors, and shuffle data over the network for the group-by &mdash; costs a single-machine tool never pays. Below a few gigabytes, pandas / Polars / DuckDB win. Spark only pays off when the data is genuinely too big for one machine.

</details>

**Problem 2.** In PySpark you run a chain of filter and select calls and notice nothing seems to happen &mdash; no time passes &mdash; until you call .count(), which suddenly takes a while. Explain.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize filter and select as transformations. — _Transformations are lazy: they only extend the plan (the DAG), they do not move or compute any data._
- Recognize count() as an action. — _An action is what triggers Spark to actually execute the whole accumulated plan._
- Conclude that all the work happens at the action. — _Spark optimizes and runs the entire chain at once when the action fires, which is why that single call takes the time._

**Answer:** Spark uses lazy evaluation. The filter and select calls are transformations &mdash; they only build up the plan and run nothing. The first action, .count(), triggers Spark to execute the whole optimized plan, so all the real work (and time) lands there. This lets Spark optimize the entire query before any data moves.

</details>

**Problem 3.** A job filters a billion-row DataFrame down to a few hundred rows, then crashes the driver with an out-of-memory error on the line rows = df.collect() &mdash; even though the final result is tiny. What is the trap, and how would you guard against it generally?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check what collect() does. — _collect() pulls every row of the DataFrame it is called on back to the single driver program._
- Check whether the filter actually shrank what collect sees. — _If collect() is called before the filter materializes — or on the wrong DataFrame — it can still try to drag the full billion rows to the driver._
- Make sure you only collect after aggregating to a small result, or avoid collect entirely. — _Pulling a small result is safe; use show()/take(n) to peek, or write() the full output to storage instead._

**Answer:** collect() pulls all its rows to the single driver, which has only one machine's memory. If it runs against the big DataFrame (rather than the tiny filtered/aggregated one), the driver tries to hold a billion rows and dies. The general guard: only collect() a result you know is small &mdash; aggregate first, use show()/take(n) to inspect, or write() large output straight to storage. Never collect a big DataFrame to the driver.

</details>